## Load df's

In [1]:
from pathlib import Path

import pandas as pd

from data_cleaning.utils import peek

CLEANED_DIR = Path("cleaned_data")

In [2]:
hai_df = pd.read_csv(CLEANED_DIR / "hai_cleaned.csv", low_memory=False)
print(hai_df.shape)
peek(hai_df)

# Build the reference set of participant IDs — all other tables will be filtered to this
hai_participants = set(hai_df["participant_id"].unique())
print(f"Unique participants in HAI: {len(hai_participants)}")

(128177, 5)
Unique participants in HAI: 3775


In [3]:
participants_df = pd.read_csv(CLEANED_DIR / "participants_cleaned.csv", low_memory=False)
participants_df = participants_df[participants_df["participant_id"].isin(hai_participants)]
print(participants_df.shape)
peek(participants_df)

(3775, 6)


,participant_id,biological_sex,race,geolocation,arm_name,age
0,SDY269.SUB112836,Female,White,Georgia,Other,28.0
1,SDY269.SUB112849,Female,Black,Georgia,Other,39.0
2,SDY269.SUB112854,Male,Black,Georgia,Other,46.0
3,SDY269.SUB112860,Female,White,Georgia,Other,32.0
4,SDY269.SUB112881,Female,Black,Georgia,Other,29.0


In [4]:
flow_df = pd.read_csv(CLEANED_DIR / "flow_cleaned.csv", low_memory=False)
flow_df = flow_df[flow_df["participant_id"].isin(hai_participants)]
print(flow_df.shape)
peek(flow_df)

(136360, 8)


,participant_id,timepoint,parent_population,population_definition,name,unit,value,material
0,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+,Freq. of,WBC",live CD45+ cells,cells/uL,176.220000,blood
1,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ1: CD27,, IgD+,Freq. of,C...",Naive B cells,cells/uL,112.780800,blood
2,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ2: CD27+,, IgD+,Freq. of,...",Memory class switched + non-class switched B c...,cells/uL,12.370644,blood
3,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ3: CD27+,, IgD,Freq. of,C...",Memory class switched B cells,cells/uL,21.322620,blood
4,SDY180.SUB119292,-7.0,NaN,"WBC/CD19+SCC/CD19+/nQ4: CD27,, IgD,Freq. of,CD19+",B cells,cells/uL,29.781180,blood


In [12]:
t2019 = pd.read_parquet(CLEANED_DIR / "transcriptomics_2019_UGA_cleaned.parquet")
t2020 = pd.read_parquet(CLEANED_DIR / "transcriptomics_2020_UGA_cleaned.parquet")
tsdy2867 = pd.read_parquet(CLEANED_DIR / "transcriptomics_SDY2867_cleaned.parquet")

# Merge on shared key columns so overlapping feature cols don't get duplicated
transcriptomics = (
    t2019
    .merge(t2020, on=["participant_id"], how="outer")
    .merge(tsdy2867, on=["participant_id"], how="outer")
)
transcriptomics = transcriptomics[transcriptomics["participant_id"].isin(hai_participants)]
print(transcriptomics.shape)

(396, 244265)


## Merge one by one

In [10]:
peek(transcriptomics, cols=1000)

,participant_id,d0_ENSG00000000003,d0_ENSG00000000005,d0_ENSG00000000419,d0_ENSG00000000457,d0_ENSG00000000460,d0_ENSG00000000938,d0_ENSG00000000971,d0_ENSG00000001036,d0_ENSG00000001084,...,d0_ENSG00000067064,d0_ENSG00000067066,d0_ENSG00000067082,d0_ENSG00000067113,d0_ENSG00000067141,d0_ENSG00000067167,d0_ENSG00000067177,d0_ENSG00000067182,d0_ENSG00000067191,d0_ENSG00000067208
0,2019_UGA.ID_001,1.001284,-0.123083,0.651376,0.546033,1.491505,-0.622255,-1.007577,0.598130,0.109457,...,0.206284,1.259382,0.084370,1.235654,1.145095,1.030026,0.424866,-0.276475,0.981637,-1.075803
1,2019_UGA.ID_005,0.586238,-0.123083,-0.682261,0.408613,0.482631,-0.929775,-0.492218,0.042291,0.819701,...,0.080004,0.496142,0.043367,1.268122,1.140054,0.800730,-0.203955,-0.166525,0.041698,0.688038
2,2019_UGA.ID_008,0.268739,-0.123083,0.012367,0.200666,-0.070990,0.235228,-0.282735,0.089897,1.102819,...,0.455270,0.480993,0.180684,1.205797,0.211669,0.174918,-0.636294,0.501367,0.323900,0.628342
3,2019_UGA.ID_011,-0.473472,-0.123083,-0.478352,0.620830,1.193616,0.432613,-0.732424,0.266911,1.158720,...,0.365980,0.401397,0.474626,1.108494,0.835086,0.718693,0.006396,0.396482,0.078049,0.920719
4,2019_UGA.ID_014,-1.145122,-0.123083,-0.007073,0.811627,0.804596,0.083290,-0.379761,0.929377,0.935906,...,0.630330,0.629786,0.792228,0.659859,0.765030,0.953876,1.688684,0.491455,-0.463860,1.155711


In [6]:
merged = participants_df.merge(flow_df, on="participant_id", how="left")
print(merged.shape)

(139994, 13)


In [7]:
merged = merged.merge(hai_df, on="participant_id", how="left", suffixes=("", "_hai"))
print(merged.shape)

(1285247, 17)


In [8]:
# merged = merged.merge(transcriptomics, on="participant_id", how="left")
# print(merged.shape)
# peek(merged)